# SegmentTable の基本

このチュートリアルでは、**GWexpy** におけるセグメント（時間区間）ベースの解析コンテナである `SegmentTable` を紹介します。通常の `EventTable` とは異なり、`SegmentTable` は区間のメタデータだけでなく、対応するペイロード（TimeSeries や ASD など）を遅延ロード（Lazy loading）形式で保持することに特化しています。

In [1]:
import numpy as np
from gwpy.segments import Segment
from gwexpy.table import SegmentTable

# 1. セグメントの作成
segs = [Segment(0, 4), Segment(4, 8), Segment(8, 12)]
st = SegmentTable.from_segments(segs, label=["A", "B", "C"])
st

/home/washimi/miniforge3/envs/ws-base/lib/python3.12/site-packages/gwpy/time/__init__.py:36: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  from lal import LIGOTimeGPS


,span,label
0,"(0, 4)",A
1,"(4, 8)",B
2,"(8, 12)",C


## SegmentCell による遅延ロード

必要になるまでデータをロードしない「ペイロード列」を追加できます。これは巨大なデータのバッチ処理に非常に有効です。

In [2]:
def my_loader():
    # ロードをシミュレート
    print("Loading series...")
    from gwpy.timeseries import TimeSeries
    return TimeSeries(np.random.randn(128), sample_rate=32)

# loader（コールバック関数のリスト）を指定してペイロード列を追加
st.add_series_column("raw", loader=[my_loader]*len(st), kind="timeseries")
st

,span,label,raw
0,"(0, 4)",A,<lazy: timeseries>
1,"(4, 8)",B,<lazy: timeseries>
2,"(8, 12)",C,<lazy: timeseries>


## 行単位の処理

`SegmentTable` は `apply()` メソッドを提供し、各行を処理して新しい列として統合できます。

In [3]:
def process_row(row):
    span = row["span"]
    return {"duration": float(span[1] - span[0]), "valid": True}

st2 = st.apply(process_row)
st2.display()

,span,label,duration,valid,raw
0,"(0, 4)",A,4.0,True,<lazy: timeseries>
1,"(4, 8)",B,4.0,True,<lazy: timeseries>
2,"(8, 12)",C,4.0,True,<lazy: timeseries>


## 明示的ロードとデータ変換

`fetch()` や `materialize()` を使ってデータを明示的にロードできます。`to_pandas()` を使うと、通常の pandas DataFrame として扱えます。

In [4]:
st2.fetch()
df = st2.to_pandas()
df.head()

Loading series...
Loading series...
Loading series...


,span,label,duration,valid
0,"(0, 4)",A,4.0,True
1,"(4, 8)",B,4.0,True
2,"(8, 12)",C,4.0,True
